# Notebook 2 · Modèles (J2) · Étapes 3 et 4 : choix du modèle et entraînement

| Étape du cycle | Partie du notebook |
| --- | --- |
| 3 · Modèle | Partie 1 : référence naïve et modèles candidats |
| 4 · Entraînement | Partie 2 : validation croisée stratifiée avec et sans `PageValues`, puis `GridSearchCV` |

**Le jeu de test n'est jamais utilisé ici.** Toutes les comparaisons se font en validation croisée sur l'entraînement.

In [ ]:
# Imports (toutes ces bibliothèques sont déjà installées dans Colab)
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## 0. Mise en place (identique dans les trois notebooks)

In [ ]:
# ===================== CONFIG : identique dans les 3 notebooks =====================
# Si vous modifiez une valeur ici, modifiez-la aussi dans les deux autres notebooks.
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00468/online_shoppers_intention.csv"
LOCAL_PATH = "online_shoppers_intention.csv"   # utilisé si le fichier a été téléversé dans Colab

TARGET = "Revenue"
NUM_COLS = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "SpecialDay",
]
LEAKY_COL = "PageValues"   # calculée à partir des transactions : testée avec et sans
CAT_COLS = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
BOOL_COLS = ["Weekend"]

FEATURE_SETS = {
    "sans_PageValues": NUM_COLS + CAT_COLS + BOOL_COLS,
    "avec_PageValues": NUM_COLS + [LEAKY_COL] + CAT_COLS + BOOL_COLS,
}

# Garde-fou : les 3 notebooks doivent obtenir exactement le même découpage.
EXPECTED_SPLIT = (9764, 2441)   # (lignes d'entraînement, lignes de test)

In [ ]:
# ===================== FONCTIONS COMMUNES : identiques dans les 3 notebooks =====================
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def load_raw():
    """Étape 1 : charge le CSV officiel (fichier local, sinon page UCI, sinon téléversement)."""
    if Path(LOCAL_PATH).exists():
        return pd.read_csv(LOCAL_PATH)
    try:
        return pd.read_csv(DATA_URL)
    except Exception as err:
        print("Téléchargement impossible :", err, "-> téléversez online_shoppers_intention.csv")
        from google.colab import files
        files.upload()
        return pd.read_csv(LOCAL_PATH)


def clean(df):
    """Nettoyage sans apprentissage (aucune statistique calculée) : sûr avant le découpage."""
    df = df.copy()
    df["Month"] = df["Month"].replace({"June": "Jun"})   # seul mois écrit en entier
    df["Weekend"] = df["Weekend"].astype(int)
    df[TARGET] = df[TARGET].astype(int)
    for col in CAT_COLS:                                 # les codes sont des catégories
        df[col] = df[col].astype(str)
    return df.drop_duplicates().reset_index(drop=True)   # avant le découpage, sinon fuite


def split(df):
    """Découpage stratifié, toujours avec la même graine."""
    X = df.drop(columns=TARGET)
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    assert (len(X_train), len(X_test)) == EXPECTED_SPLIT, "Découpage différent des autres notebooks !"
    return X_train, X_test, y_train, y_test


def make_preprocessor(feature_set):
    """Étape 2 : à n'ajuster que sur l'entraînement (ou dans un Pipeline)."""
    num = [c for c in FEATURE_SETS[feature_set] if c in NUM_COLS + [LEAKY_COL]]
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
        ("bool", "passthrough", BOOL_COLS),
    ], remainder="drop")   # remainder="drop" retire PageValues dans la variante « sans »


def build_estimator(name, params=None):
    """Étape 3 : les modèles candidats."""
    estimator = {
        "Reference_naive": DummyClassifier(strategy="most_frequent"),
        "Regression_logistique": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "Foret_aleatoire": RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "Gradient_boosting": HistGradientBoostingClassifier(
            class_weight="balanced", random_state=RANDOM_STATE),
    }[name]
    if params:
        estimator.set_params(**params)
    return estimator


def make_pipeline(name, feature_set, params=None):
    return Pipeline([("prep", make_preprocessor(feature_set)),
                     ("model", build_estimator(name, params))])


CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
df = clean(load_raw())
X_train, X_test, y_train, y_test = split(df)
print("Entraînement :", X_train.shape, "| Test :", X_test.shape,
      "| taux d'achat train / test :", round(y_train.mean(), 3), "/", round(y_test.mean(), 3))

# Partie 1 · Modèle

| Modèle | Pourquoi l'essayer |
| --- | --- |
| Référence naïve | Prédit toujours « pas d'achat » : le score à battre |
| Régression logistique | Simple, rapide, interprétable |
| Forêt aléatoire | Capte les interactions sans normalisation |
| Gradient boosting | Souvent le plus performant sur des données tabulaires |

Tous les modèles (sauf la référence) utilisent `class_weight="balanced"` à cause des 15 % d'achats.

In [ ]:
MODELS = ["Reference_naive", "Regression_logistique", "Foret_aleatoire", "Gradient_boosting"]
SCORING = {"f1": "f1", "rappel": "recall", "precision": "precision", "auc": "roc_auc"}

# Partie 2 · Entraînement

### 2.1 Validation croisée de tous les modèles, avec et sans `PageValues`

In [ ]:
from sklearn.model_selection import cross_validate

rows = []
for feature_set in FEATURE_SETS:
    for name in MODELS:
        scores = cross_validate(make_pipeline(name, feature_set), X_train, y_train,
                                cv=CV, scoring=SCORING, n_jobs=-1)
        row = {"variables": feature_set, "modele": name}
        for metric in SCORING:
            values = scores[f"test_{metric}"]
            row[metric] = f"{values.mean():.3f} ± {values.std():.3f}"
            row[f"_{metric}"] = values.mean()
        rows.append(row)
        print(f"{feature_set:16s} {name:22s} F1 = {row['f1']}")

results = pd.DataFrame(rows)
display(results[["variables", "modele", "f1", "rappel", "precision", "auc"]])

In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(x="_f1", y="modele", hue="variables", data=results)
plt.xlabel("F1 « achat » en validation croisée")
plt.ylabel("")
plt.title("Avec ou sans PageValues")
plt.show()

### 2.2 Lecture des résultats (à rédiger)

- *Écart avec / sans `PageValues` :* …
- *Ce que cela dit sur la fuite possible :* …
- *Modèle le plus prometteur pour chaque variante :* …

### 2.3 Réglage des hyperparamètres avec `GridSearchCV`

In [ ]:
from sklearn.model_selection import GridSearchCV

PARAM_GRIDS = {
    "Regression_logistique": {"model__C": [0.01, 0.1, 1, 10]},
    "Foret_aleatoire": {"model__max_depth": [None, 10, 20], "model__min_samples_leaf": [1, 5]},
    "Gradient_boosting": {"model__learning_rate": [0.05, 0.1], "model__max_depth": [None, 6]},
}

best = {}
for feature_set in FEATURE_SETS:
    subset = results[(results.variables == feature_set) & (results.modele != "Reference_naive")]
    name = subset.sort_values("_f1", ascending=False).iloc[0]["modele"]
    search = GridSearchCV(make_pipeline(name, feature_set), PARAM_GRIDS[name],
                          scoring="f1", cv=CV, n_jobs=-1)
    search.fit(X_train, y_train)
    params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
    best[feature_set] = {"model": name, "params": params, "cv_f1": round(float(search.best_score_), 3)}
    print(f"{feature_set:16s} {name:22s} F1 CV = {search.best_score_:.3f} | {params}")

### 2.4 Configuration à recopier dans le notebook 3

Copiez le dictionnaire affiché ci-dessous dans la cellule `BEST` du notebook 3.

In [ ]:
print("BEST =", best)

## Bilan J2 (à rédiger)

- *Modèle retenu pour chaque variante et pourquoi :* …
- *Hyperparamètres trouvés :* …